# Session matrix analysis (`run_experiment_matrix.sh`)

Batch runs usually land under one of these layouts:

- **`logs_exp/session_YYYYMMDD_HHMMSS/<run_label>/`**
- **`logs_exp/log/session_YYYYMMDD_HHMMSS/<run_label>/`**

This notebook:

1. **Resolves** a session directory (`logs_exp/.last_session`, env `SESSION_DIR`, manual path, or latest `session_*`).
2. **Lists** child folders (`phase1_*`, `phase2_*`, `phase3_*`).
3. **Templates** for **`parse_logs.load_pull_log` / `load_labeled_vm_runs` / `load_phase2_triple`** and **`[utility_mode] adaptive`** scraping.
4. Provides a **single-run quick path** (useful when session only has `phase3_delay_auto`).

**Prereqs:** `pandas`, `matplotlib`. Run from repo root or keep `REPO` below correct.

In [2]:
from __future__ import annotations

import os
import re
import sys
from pathlib import Path

import pandas as pd


def find_repo(start: Path | None = None) -> Path:
    """Walk up from start until scripts/analyze/parse_logs.py exists."""
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "scripts" / "analyze" / "parse_logs.py").is_file():
            return p.resolve()
    raise FileNotFoundError(
        "Could not find repo root. Open Jupyter from repo root or set REPO manually."
    )


REPO = find_repo()
# REPO = Path("/home/mininet/Project/4D-MAP")  # uncomment if find_repo() fails

sys.path.insert(0, str(REPO / "scripts" / "analyze"))
from parse_logs import (
    estimate_tc_pull_offset_seconds,
    load_labeled_vm_runs,
    load_phase2_triple,
    load_pull_log,
    load_tc_log,
)

print("REPO =", REPO)

REPO = /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment


## 1. Resolve `SESSION` (pick one strategy)

Uncomment **one** block in the next cell.

In [3]:
LOGS_EXP = REPO / "logs_exp"


def _session_candidates(logs_exp: Path) -> list[Path]:
    """Collect session directories from both common layouts."""
    cands: list[Path] = []
    cands.extend([p for p in logs_exp.glob("session_*") if p.is_dir()])
    cands.extend([p for p in (logs_exp / "log").glob("session_*") if p.is_dir()])
    # de-dup and newest first
    uniq = {p.resolve(): p.resolve() for p in cands}
    return sorted(uniq.keys(), key=lambda x: x.stat().st_mtime, reverse=True)


def resolve_session(logs_exp: Path = LOGS_EXP) -> Path | None:
    """Try: env SESSION_DIR → .last_session file → newest session_* directory."""
    env = os.environ.get("SESSION_DIR", "").strip()
    if env:
        p = Path(env)
        if not p.is_absolute():
            p = REPO / p
        if p.is_dir():
            return p.resolve()

    last = logs_exp / ".last_session"
    if last.is_file():
        rel = last.read_text(encoding="utf-8").strip().splitlines()[0]
        p = REPO / rel if not Path(rel).is_absolute() else Path(rel)
        if p.is_dir():
            return p.resolve()

    sessions = _session_candidates(logs_exp)
    return sessions[0] if sessions else None


# --- choose one ----------------------------------------------------------
SESSION = resolve_session()
# SESSION = REPO / "logs_exp" / "session_20260405_185156"      # layout A
# SESSION = REPO / "logs_exp" / "log" / "session_20260405_191618"  # layout B
# ------------------------------------------------------------------------

assert SESSION is not None and SESSION.is_dir(), f"No session dir under {LOGS_EXP}"
print("SESSION =", SESSION)

SESSION = /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment/logs_exp/log/session_20260406_125609


## 2. Traverse run folders under the session

Each child of `SESSION` is one **`--run-label`** (e.g. `phase1_default_T`).

In [4]:
def list_session_runs(session: Path, prefix: str | None = None) -> list[Path]:
    """Sorted child dirs; optional filter e.g. 'phase1', 'phase2_delay', 'phase3'."""
    runs = [p for p in session.iterdir() if p.is_dir()]
    runs.sort(key=lambda p: p.name)
    if prefix:
        runs = [p for p in runs if p.name.startswith(prefix)]
    return runs


def pull_path(run_dir: Path) -> Path | None:
    g = list(run_dir.glob("pull_*.log"))
    return g[0] if g else None


def summarize_session(session: Path) -> pd.DataFrame:
    rows = []
    for d in list_session_runs(session):
        pl = pull_path(d)
        rows.append(
            {
                "run_label": d.name,
                "pull_log": pl.name if pl else None,
                "has_tc_delay": bool(list(d.glob("tc_delay_*.log"))),
                "has_tc_loss": bool(list(d.glob("tc_loss_*.log"))),
            }
        )
    return pd.DataFrame(rows)


df_runs = summarize_session(SESSION)
display(df_runs) if "display" in dir() else print(df_runs.to_string())

           run_label                  pull_log  has_tc_delay  has_tc_loss
0  phase3_delay_auto  pull_20260406_125631.log          True        False


## 3. Phase 1 template — same scenario, `baseline` / `T` / `D` / `L`

Adjust **`scenario`** to match your matrix (`default`, `t`, `d`, `l`, …).

In [5]:
scenario = "default"
suffixes = ("baseline", "T", "D", "L")
label_to_dir: dict[str, Path] = {}
for suf in suffixes:
    name = f"phase1_{scenario}_{suf}"
    d = SESSION / name
    if d.is_dir() and pull_path(d):
        label_to_dir[suf] = d
    else:
        print(f"skip missing: {name}")

df_util_p1 = df_mon_p1 = pd.DataFrame()
tc_p1: dict = {}
if len(label_to_dir) >= 2:
    df_util_p1, df_mon_p1, tc_p1 = load_labeled_vm_runs(label_to_dir, tc_from_label=None)
    print("Phase1 utility rows:", len(df_util_p1), "monitor rows:", len(df_mon_p1))
    print(df_util_p1["label"].value_counts())
else:
    print("Need at least two phase1_* dirs with pull logs; got:", list(label_to_dir))

skip missing: phase1_default_baseline
skip missing: phase1_default_T
skip missing: phase1_default_D
skip missing: phase1_default_L
Need at least two phase1_* dirs with pull logs; got: []


## 4. Phase 2 template — `phase2_delay_*` with tc alignment

Pick one utility (e.g. `T`) and align **`tc_delay_*.log`** to pull time axis via `estimate_tc_pull_offset_seconds`.

In [6]:
um = "T"
run_delay = SESSION / f"phase2_delay_{um}"
pull = pull_path(run_delay)
tc_files = list(run_delay.glob("tc_delay_*.log"))
if pull and tc_files:
    df_u, df_m = load_pull_log(pull, label=f"delay_{um}")
    tc_df = load_tc_log(tc_files[0])
    off = estimate_tc_pull_offset_seconds(pull, tc_files[0])
    print("tc steps:", len(tc_df), "offset_sec", off)
    display(tc_df.head()) if "display" in dir() else print(tc_df.head())
else:
    print("No phase2_delay_* run or missing pull/tc in", run_delay)

No phase2_delay_* run or missing pull/tc in /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment/logs_exp/log/session_20260406_125609/phase2_delay_T


## 5. Phase 2 triple — baseline / delay / loss (same utility)

Uses **`load_phase2_triple`** when the three folders exist (names from `run_experiment_matrix.sh`).

In [7]:
um = "T"
d_base = SESSION / f"phase2_baseline_{um}"
d_del = SESSION / f"phase2_delay_{um}"
d_loss = SESSION / f"phase2_loss_{um}"
if all(p.is_dir() and pull_path(p) for p in (d_base, d_del, d_loss)):
    df_u3, df_m3, tc_steps = load_phase2_triple(d_base, d_del, d_loss)
    print("combined utility:", len(df_u3), "tc keys:", list(tc_steps.keys()))
else:
    print("Missing one of phase2_baseline/delay/loss for utility", um)

Missing one of phase2_baseline/delay/loss for utility T


## 6. Phase 3 — `auto` + adaptive mode transitions

Scheduler logs **`[utility_mode] adaptive X -> Y (...)`** (rate-limited). Below: regex scrape into a small table.

In [8]:
_RE_ADAPTIVE = re.compile(
    r"\[utility_mode\] adaptive (?P<frm>\w+) -> (?P<to>\w+) "
    r"\(max_loss=(?P<ml>[^\s]+) max_owd_ms=(?P<owd>[^)]+)\)"
)


def parse_adaptive_transitions(pull_log: Path) -> pd.DataFrame:
    rows = []
    with open(pull_log, encoding="utf-8", errors="replace") as f:
        for line in f:
            m = _RE_ADAPTIVE.search(line)
            if m:
                rows.append(m.groupdict() | {"line": line.strip()[:200]})
    return pd.DataFrame(rows)


p3 = SESSION / "phase3_delay_auto"
pl = pull_path(p3)
if pl:
    df_adapt = parse_adaptive_transitions(pl)
    print("adaptive transitions:", len(df_adapt))
    display(df_adapt) if "display" in dir() else print(df_adapt)
else:
    print("No phase3_delay_auto or pull log; try phase3_static_*_auto")

adaptive transitions: 0
Empty DataFrame
Columns: []
Index: []


## 7. Single-run quick analysis (`phase3_delay_auto`)

Use this when the session has only one run folder (common for a manual one-shot test).

**Prerequisite:** run **§6** (Phase 3 regex + `parse_adaptive_transitions`) first, or the code cell below will fail with `NameError`.

The cell below auto-picks:

1. `phase3_delay_auto` if present,
2. otherwise the only run folder,
3. otherwise the first run folder.

It also prints **`[meta] utility_mode=...`** from `pull_*.log`. If the folder name contains `auto` but meta shows `T`, your **`4dmap` binary likely was not rebuilt** with `auto` support — adaptive lines will be empty.


In [9]:
single_run = None
run_dirs = list_session_runs(SESSION)

preferred = SESSION / "phase3_delay_auto"
if preferred.is_dir():
    single_run = preferred
elif len(run_dirs) == 1:
    single_run = run_dirs[0]
elif run_dirs:
    single_run = run_dirs[0]

if single_run is None:
    print("No run directory found under", SESSION)
else:
    print("single_run =", single_run)
    pl = pull_path(single_run)
    if pl is None:
        print("No pull_*.log found in", single_run)
    else:
        meta_um = None
        with open(pl, encoding="utf-8", errors="replace") as f:
            for line in f:
                if "[meta]" in line and "utility_mode=" in line:
                    mm = re.search(r"utility_mode=(\S+)", line)
                    meta_um = mm.group(1) if mm else None
                    break

        print("[meta] utility_mode (from pull, first line):", meta_um)
        if (
            "auto" in single_run.name.lower()
            and meta_um
            and str(meta_um).upper() not in ("AUTO", "ADAPT")
        ):
            print(
                "WARNING: folder name suggests auto, but [meta] is not auto/adapt. "
                "Rebuild ./4dmap from current branch and re-run the experiment."
            )

        df_util_single, df_mon_single = load_pull_log(pl, label=single_run.name)
        print("utility rows:", len(df_util_single), "monitor rows:", len(df_mon_single))
        if not df_util_single.empty:
            print("mode counts:")
            print(df_util_single["mode"].value_counts(dropna=False))

        df_adapt_single = parse_adaptive_transitions(pl)
        print("adaptive transitions:", len(df_adapt_single))
        if not df_adapt_single.empty:
            display(df_adapt_single.head(20)) if "display" in dir() else print(df_adapt_single.head(20))

        tc_delay = next(single_run.glob("tc_delay_*.log"), None)
        tc_loss = next(single_run.glob("tc_loss_*.log"), None)
        if tc_delay:
            tc_df_single = load_tc_log(tc_delay)
            off_single = estimate_tc_pull_offset_seconds(pl, tc_delay)
            print("tc_delay steps:", len(tc_df_single), "offset_sec:", off_single)
            display(tc_df_single) if "display" in dir() else print(tc_df_single)
        elif tc_loss:
            tc_df_single = load_tc_log(tc_loss)
            off_single = estimate_tc_pull_offset_seconds(pl, tc_loss)
            print("tc_loss steps:", len(tc_df_single), "offset_sec:", off_single)
            display(tc_df_single) if "display" in dir() else print(tc_df_single)
        else:
            print("No tc_delay_*.log / tc_loss_*.log in", single_run)

SyntaxError: invalid syntax (3414061684.py, line 14)

## 8. Stub plot (Phase 1 OWD per path)

Replace with your styling; or call **`python scripts/analyze/plot_phase2.py`** on selected dirs.

In [ ]:
import matplotlib.pyplot as plt

if "df_util_p1" in dir() and df_util_p1 is not None and not df_util_p1.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    for label in sorted(df_util_p1["label"].unique()):
        for path in (0, 1):
            sub = df_util_p1[(df_util_p1["label"] == label) & (df_util_p1["path"] == path)]
            if sub.empty:
                continue
            ax.plot(sub["t"], sub["owd_ms"], label=f"{label} path{path}", alpha=0.8)
    ax.set_xlabel("t (s from first utility line)")
    ax.set_ylabel("owd_ms ([utility] line)")
    ax.legend(fontsize=7, ncol=2)
    ax.set_title(f"Phase1 scenario={scenario} — OWD")
    plt.tight_layout()
    plt.show()
else:
    print("Run Phase 1 cell first to define df_util_p1.")